# 3. Import Libraries

In [18]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.tools import Tool
from langchain_classic.agents import initialize_agent, AgentType
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from sqlalchemy import create_engine
import sqlite3

# Part 1: Create SQL Database

In [19]:
conn = sqlite3.connect("../Gen_ai_data/company.db")

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS employees(
id INTEGER PRIMARY KEY,
name TEXT,
department TEXT,
salary INTEGER
)""")

cursor.executemany("""INSERT INTO employees(name, department, salary)
VALUES(?,?,?)""",[("John", "AI", 90000),
                 ("Sara", "ML", 95000),
                 ("Mike", "Data", 80000),
                 ("Anna", "AI", 105000)
                 ])
conn.commit()
conn.close()

# Connect Database With Langchain

In [20]:
engine = create_engine("sqlite:///../Gen_ai_data/company.db")

db = SQLDatabase(engine)

# Par2: Build Rag System

In [23]:
loader = PyMuPDFLoader("../Gen_ai_data/sample_ai_document.pdf")

docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

chunks = splitter.split_documents(docs)

# Create Embeddings

In [26]:
embeddings = OllamaEmbeddings(
    model = "nomic-embed-text"
)

# Create vector Database

In [27]:
vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

retriever = vectorstore.as_retriever()

# Part 3: Crreate Tools

# Tool 1: --- Rag Retriever

In [28]:
def rag_tool(query):
    docs = retriever.invoke(query)

    return docs[0].page_content

In [29]:
rag = Tool(
    name = "Rag_Document_Search",
    func = rag_tool,
    description = "Searches knowledge base documents"
)

# Tool 2: SQL Database Tool

In [30]:
toolkit = SQLDatabaseToolkit(db=db)

sql_tools = toolkit.get_tools()

ValidationError: 1 validation error for SQLDatabaseToolkit
llm
  Field required [type=missing, input_value={'db': <langchain_communi... at 0x00000215B3C4D9D0>}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing

# Tool 3 Calculator

In [31]:
def calculator(query):
    try:
        return str(eval(query))
    except:
        return "Invalid math expression"

In [32]:
calculator_tool = Tool(
    name = "Calculator",
    func = calculator,
    description = "Useful for math Calculations"
)